# Dentalyze Care — Faster R-CNN Model Training on Google Colab

This notebook trains a **Faster R-CNN (ResNet50-FPN)** deep learning model on the **DENTEX Dental X-Ray Dataset** to detect 4 conditions:
1. **Caries**
2. **Deep Caries**
3. **Periapical Lesion (Abscess)**
4. **Impacted Tooth**

### Instructions:
1. In Google Colab, select **Runtime > Change runtime type > T4 GPU** (or A100/L4).
2. Run all cells sequentially.
3. At the end, download `dentex_frcnn_best.pth` and place it in your local `backend/trained_models/` folder.

In [ ]:
# Step 1: Check GPU Acceleration
!nvidia-smi
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 2: Install required packages
!pip install -q torchvision opencv-python-headless matplotlib tqdm scikit-learn Pillow

In [ ]:
# Step 3: Mount Google Drive & Extract DENTEX Dataset
import os
import zipfile
import shutil
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

drive_folder = '/content/drive/MyDrive/New-FYP-DentalyzeCare-work-2026'
dest_dir = '/content/dentex_data'
os.makedirs(dest_dir, exist_ok=True)

print(f"Searching for dataset in: {drive_folder}...")

# 1. Check for zip files (dataset 101.zip, dataset_101.zip, etc.)
candidate_zips = [
    os.path.join(drive_folder, 'dataset 101.zip'),
    os.path.join(drive_folder, 'dataset_101.zip'),
    '/content/drive/MyDrive/dataset 101.zip',
    '/content/drive/MyDrive/dataset_101.zip',
]

# Search inside drive_folder for any .zip file
if os.path.exists(drive_folder):
    for root, dirs, files in os.walk(drive_folder):
        for f in files:
            if f.endswith('.zip') and ('101' in f or 'dataset' in f.lower() or 'dentex' in f.lower() or 'training' in f.lower()):
                full_p = os.path.join(root, f)
                if full_p not in candidate_zips:
                    candidate_zips.insert(0, full_p)

found_zip = next((p for p in candidate_zips if os.path.isfile(p)), None)

if found_zip:
    print(f"[OK] Found zip archive: {found_zip}")
    print(f"Extracting to fast local SSD ({dest_dir})...")
    with zipfile.ZipFile(found_zip, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)
    print("[OK] Extraction complete!")
else:
    # 2. If it is already uncompressed as a folder (e.g., 'dataset 101' or 'dataset 101.zip' as a folder)
    candidate_folders = [
        os.path.join(drive_folder, 'dataset 101.zip'),
        os.path.join(drive_folder, 'dataset 101'),
        os.path.join(drive_folder, 'dataset_101'),
        drive_folder,
    ]
    found_folder = None
    for fld in candidate_folders:
        if os.path.isdir(fld):
            # Check if this folder contains training_data or images
            for root, dirs, files in os.walk(fld):
                if 'train_quadrant_enumeration_disease.json' in files:
                    found_folder = root if 'training_data' not in root else root.split('training_data')[0]
                    break
            if found_folder:
                break
    
    if found_folder and os.path.exists(found_folder):
        print(f"[OK] Found dataset folder: {found_folder}")
        print(f"Copying to fast local SSD ({dest_dir})...")
        !cp -r "$found_folder"/* /content/dentex_data/
        print("[OK] Copy complete!")
    else:
        print(f"[!] Could not locate dataset in {drive_folder}. Contents of folder:")
        if os.path.exists(drive_folder):
            print(os.listdir(drive_folder))
        else:
            print(f"Directory does not exist: {drive_folder}")

# Handle nested directory structure (e.g. dest_dir/'dataset 101' or dest_dir/'dataset 101.zip')
for item in os.listdir(dest_dir):
    nested_item = os.path.join(dest_dir, item)
    if os.path.isdir(nested_item) and item.startswith('dataset'):
        for sub in os.listdir(nested_item):
            s = os.path.join(nested_item, sub)
            d = os.path.join(dest_dir, sub)
            if not os.path.exists(d):
                shutil.move(s, dest_dir)

annotation_dest = '/content/dentex_data/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json'
images_dir = '/content/dentex_data/training_data/quadrant-enumeration-disease/xrays'

if os.path.exists(annotation_dest) and os.path.exists(images_dir):
    img_count = len(os.listdir(images_dir))
    print(f"[OK] Dataset verified successfully! {img_count} images found.")
    print(f"[OK] Annotations file verified: {annotation_dest}")
else:
    print("[!] Current /content/dentex_data layout:")
    !find /content/dentex_data -maxdepth 3


In [ ]:
# Step 4: Define Dataset, Model Architecture, and Training Loop
import os
import json
import random
import numpy as np
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from sklearn.model_selection import train_test_split
from tqdm import tqdm

DISEASE_CLASSES = {
    0: "background",
    1: "Caries",
    2: "Deep Caries",
    3: "Periapical Lesion",
    4: "Impacted Tooth",
}
NUM_CLASSES = 5

class DentexDataset(Dataset):
    def __init__(self, images_dir, annotations, image_ids=None, transforms=None):
        self.images_dir = images_dir
        self.transforms = transforms
        self.image_map = {img["id"]: img for img in annotations["images"]}
        self.img_to_anns = {}
        for ann in annotations["annotations"]:
            iid = ann["image_id"]
            if image_ids is None or iid in image_ids:
                self.img_to_anns.setdefault(iid, []).append(ann)
        self.ids = [iid for iid in (image_ids if image_ids else self.img_to_anns.keys()) if iid in self.img_to_anns]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        img_info = self.image_map[image_id]
        img_path = os.path.join(self.images_dir, img_info["file_name"])
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        boxes = []
        labels = []
        for ann in self.img_to_anns[image_id]:
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0:
                continue
            boxes.append([x, y, x + w, y + h])
            cat_id = ann.get("category_id_3", ann.get("category_id", 0))
            labels.append(cat_id + 1)

        img = img.astype(np.float32) / 255.0
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([image_id])}
        img_tensor = torch.tensor(img).permute(2, 0, 1)
        return img_tensor, target

def collate_fn(batch):
    return tuple(zip(*batch))

def create_model(num_classes=NUM_CLASSES):
    model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

print("Model and Dataset definitions ready!")

In [ ]:
# Step 5: Train Faster R-CNN Model
import os
import json
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

annotation_dest = '/content/dentex_data/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json'
images_dir = '/content/dentex_data/training_data/quadrant-enumeration-disease/xrays'
output_dir = "/content/trained_models"
os.makedirs(output_dir, exist_ok=True)

if os.path.exists(annotation_dest) and os.path.exists(images_dir):
    with open(annotation_dest, "r") as f:
        annotations = json.load(f)

    all_image_ids = [img["id"] for img in annotations["images"]]
    annotated_ids = set(ann["image_id"] for ann in annotations["annotations"])
    valid_ids = [iid for iid in all_image_ids if iid in annotated_ids]
    train_ids, val_ids = train_test_split(valid_ids, test_size=0.2, random_state=42)

    print(f"Total annotated images: {len(valid_ids)}")
    print(f"Train split: {len(train_ids)} | Validation split: {len(val_ids)}")

    train_dataset = DentexDataset(images_dir, annotations, image_ids=set(train_ids))
    val_dataset = DentexDataset(images_dir, annotations, image_ids=set(val_ids))

    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

    model = create_model(NUM_CLASSES).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.0005)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.5)

    epochs = 25
    best_val_loss = float("inf")

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            optimizer.zero_grad()
            losses.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            optimizer.step()
            running_loss += losses.item()

        scheduler.step()
        avg_loss = running_loss / max(1, len(train_loader))
        print(f"Epoch {epoch+1} Train Loss: {avg_loss:.4f}")

        # Save checkpoint
        if avg_loss < best_val_loss:
            best_val_loss = avg_loss
            torch.save(model.state_dict(), f"{output_dir}/dentex_frcnn_best.pth")
            # Also save backup copy to Google Drive folder
            drive_target_dir = "/content/drive/MyDrive/New-FYP-DentalyzeCare-work-2026"
            os.makedirs(drive_target_dir, exist_ok=True)
            drive_save_path = os.path.join(drive_target_dir, "dentex_frcnn_best.pth")
            try:
                torch.save(model.state_dict(), drive_save_path)
                print(f"  [OK] Saved best model and backed up to Google Drive ({drive_save_path})!")
            except Exception:
                try:
                    torch.save(model.state_dict(), "/content/drive/MyDrive/dentex_frcnn_best.pth")
                    print("  [OK] Saved best model to Google Drive (MyDrive/dentex_frcnn_best.pth)!")
                except Exception:
                    pass
            print(f"  [OK] Saved best model checkpoint to {output_dir}/dentex_frcnn_best.pth")
else:
    print("[!] Dataset or annotations not found at /content/dentex_data.")
    print("Please run Step 3 first to extract the dataset from Google Drive.")


In [ ]:
# Step 6: Download the trained checkpoint
import os
from google.colab import files
best_pth = "/content/trained_models/dentex_frcnn_best.pth"
if os.path.exists(best_pth):
    files.download(best_pth)
    print("[OK] Download initiated!")
    print("[OK] A backup copy is also saved in: My Drive/New-FYP-DentalyzeCare-work-2026/dentex_frcnn_best.pth")
    print("Move this file into backend/trained_models/dentex_frcnn_best.pth on your local machine.")
else:
    print("Checkpoint file not found yet. Run Step 5 first.")


In [ ]:
# Step 7: Evaluate Model Performance & Compute mAP Metrics (Step B)
import os
import json
import numpy as np
import torch
from PIL import Image

print('--- Evaluating Faster R-CNN Model Performance ---')
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    if inter <= 0: return 0.0
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

# Evaluate on val_loader
class_names = {1: 'Impacted Tooth', 2: 'Dental Caries', 3: 'Periapical Lesion', 4: 'Deep Caries'}
class_stats = {cid: {'tp': 0, 'fp': 0, 'fn': 0, 'gt': 0, 'ious': []} for cid in range(1, 5)}
all_ious = []

print(f'Running inference on validation set using {device}...')
with torch.no_grad():
    for images, targets in val_loader:
        images = [img.to(device) for img in images]
        predictions = model(images)
        for pred, target in zip(predictions, targets):
            p_boxes = pred['boxes'].cpu().numpy()
            p_labels = pred['labels'].cpu().numpy()
            p_scores = pred['scores'].cpu().numpy()
            
            mask = p_scores >= 0.25
            p_boxes, p_labels, p_scores = p_boxes[mask], p_labels[mask], p_scores[mask]
            
            g_boxes = target['boxes'].cpu().numpy()
            g_labels = target['labels'].cpu().numpy()
            
            for cid in range(1, 5):
                c_p_idx = np.where(p_labels == cid)[0]
                c_g_idx = np.where(g_labels == cid)[0]
                class_stats[cid]['gt'] += len(c_g_idx)
                
                matched_g = set()
                for pi in c_p_idx:
                    best_iou, best_gi = 0.0, -1
                    for gi in c_g_idx:
                        if gi in matched_g: continue
                        iou = compute_iou(p_boxes[pi], g_boxes[gi])
                        if iou > best_iou:
                            best_iou, best_gi = iou, gi
                    if best_iou >= 0.5 and best_gi != -1:
                        class_stats[cid]['tp'] += 1
                        class_stats[cid]['ious'].append(best_iou)
                        all_ious.append(best_iou)
                        matched_g.add(best_gi)
                    else:
                        class_stats[cid]['fp'] += 1
                class_stats[cid]['fn'] += len(c_g_idx) - len(matched_g)

print('\n=================== EVALUATION RESULTS ===================')
precs, recs = [], []
for cid, name in class_names.items():
    st = class_stats[cid]
    p = st['tp'] / (st['tp'] + st['fp']) if (st['tp'] + st['fp']) > 0 else 0.0
    r = st['tp'] / (st['tp'] + st['fn']) if (st['tp'] + st['fn']) > 0 else 0.0
    f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
    avg_iou = np.mean(st['ious']) if st['ious'] else 0.0
    precs.append(p); recs.append(r)
    print(f'{name:<20} | Precision: {p:<6.1%} | Recall: {r:<6.1%} | F1: {f1:<6.1%} | Avg IoU: {avg_iou:<6.3f}')
print('-' * 70)
print(f'Mean IoU: {np.mean(all_ious):.3f}')
print(f'mAP @ IoU 0.50: {np.mean(precs):.1%}')
print('==========================================================')
